In [ ]:
# kernels.py
import torch
import triton
import triton.language as tl


# ============================================================
# Contiguous Decode Attention Kernel
# Each request owns a contiguous KV cache:
# K: [B, H, MAX_SEQ_LEN, D]
# V: [B, H, MAX_SEQ_LEN, D]
# Q: [B, H, D]
# O: [B, H, D]
# ============================================================

@triton.jit
def contiguous_decode_attn_kernel(
    Q, K, V, O,
    CONTEXT_LENS,
    stride_qb: tl.constexpr,
    stride_qh: tl.constexpr,
    stride_kb: tl.constexpr,
    stride_kh: tl.constexpr,
    stride_ks: tl.constexpr,
    stride_vb: tl.constexpr,
    stride_vh: tl.constexpr,
    stride_vs: tl.constexpr,
    stride_ob: tl.constexpr,
    stride_oh: tl.constexpr,
    MAX_SEQ_LEN: tl.constexpr,
    D: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    b = tl.program_id(0)
    h = tl.program_id(1)

    offs_d = tl.arange(0, D)

    # TODO 1:
    # load q vector: q = Q[b, h, :]
    q = tl.load(
        Q + b * stride_qb + h * stride_qh + offs_d
    )

    context_len = tl.load(CONTEXT_LENS + b)

    m_i = -float("inf")
    l_i = 0.0
    acc = tl.zeros((D,), dtype=tl.float32)

    # TODO 2:
    # loop over KV tokens in blocks of BLOCK_N
    for start_n in range(0, MAX_SEQ_LEN, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)
        mask_n = offs_n < context_len

        # TODO 3:
        # load K block: shape [BLOCK_N, D]
        k = tl.load(
            K
            + b * stride_kb
            + h * stride_kh
            + offs_n[:, None] * stride_ks
            + offs_d[None, :],
            mask=mask_n[:, None],
            other=0.0,
        )

        # TODO 4:
        # scores = dot(q, k)
        scores = tl.sum(k * q[None, :], axis=1)

        scores = tl.where(mask_n, scores, -float("inf"))

        # Online softmax update
        m_new = tl.maximum(m_i, tl.max(scores, axis=0))
        p = tl.exp(scores - m_new)
        alpha = tl.exp(m_i - m_new)

        # TODO 5:
        # load V block
        v = tl.load(
            V
            + b * stride_vb
            + h * stride_vh
            + offs_n[:, None] * stride_vs
            + offs_d[None, :],
            mask=mask_n[:, None],
            other=0.0,
        )

        # TODO 6:
        # update accumulator
        acc = acc * alpha + tl.sum(p[:, None] * v, axis=0)
        l_i = l_i * alpha + tl.sum(p, axis=0)
        m_i = m_new

    out = acc / l_i

    tl.store(
        O + b * stride_ob + h * stride_oh + offs_d,
        out
    )


# ============================================================
# Paged Decode Attention Kernel
# Each request owns logical blocks.
# block_table[b, logical_block_id] -> physical_block_id
#
# K_CACHE: [NUM_BLOCKS, H, BLOCK_SIZE, D]
# V_CACHE: [NUM_BLOCKS, H, BLOCK_SIZE, D]
# Q:       [B, H, D]
# O:       [B, H, D]
# ============================================================

@triton.jit
def paged_decode_attn_kernel(
    Q, K_CACHE, V_CACHE, BLOCK_TABLE, O,
    CONTEXT_LENS,
    stride_qb: tl.constexpr,
    stride_qh: tl.constexpr,
    stride_kblock: tl.constexpr,
    stride_kh: tl.constexpr,
    stride_kt: tl.constexpr,
    stride_vblock: tl.constexpr,
    stride_vh: tl.constexpr,
    stride_vt: tl.constexpr,
    stride_ob: tl.constexpr,
    stride_oh: tl.constexpr,
    stride_bt_b: tl.constexpr,
    stride_bt_l: tl.constexpr,
    MAX_BLOCKS_PER_SEQ: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    D: tl.constexpr,
):
    b = tl.program_id(0)
    h = tl.program_id(1)

    offs_d = tl.arange(0, D)
    offs_t = tl.arange(0, BLOCK_SIZE)

    # TODO 1:
    # load q vector
    q = tl.load(Q + b * stride_qb + h * stride_qh + offs_d)

    context_len = tl.load(CONTEXT_LENS + b)

    m_i = -float("inf")
    l_i = 0.0
    acc = tl.zeros((D,), dtype=tl.float32)

    # TODO 2:
    # iterate over logical KV blocks
    for logical_block in range(0, MAX_BLOCKS_PER_SEQ):
        token_start = logical_block * BLOCK_SIZE
        token_ids = token_start + offs_t
        mask_t = token_ids < context_len

        # TODO 3:
        # physical_block = block_table[b, logical_block]
        physical_block = tl.load(
            BLOCK_TABLE + b * stride_bt_b + logical_block * stride_bt_l
        )

        # TODO 4:
        # load K block from physical block
        k = tl.load(
            K_CACHE
            + physical_block * stride_kblock
            + h * stride_kh
            + offs_t[:, None] * stride_kt
            + offs_d[None, :],
            mask=mask_t[:, None],
            other=0.0,
        )

        # TODO 5:
        # compute attention score
        scores = tl.sum(k * q[None, :], axis=1)
        scores = tl.where(mask_t, scores, -float("inf"))

        # TODO 6:
        # online softmax
        m_new = tl.maximum(m_i, tl.max(scores, axis=0))
        p = tl.exp(scores - m_new)
        alpha = tl.exp(m_i - m_new)

        # TODO 7:
        # load V block
        v = tl.load(
            V_CACHE
            + physical_block * stride_vblock
            + h * stride_vh
            + offs_t[:, None] * stride_vt
            + offs_d[None, :],
            mask=mask_t[:, None],
            other=0.0,
        )

        # TODO 8:
        # update acc / l / m
        acc = acc * alpha + tl.sum(p[:, None] * v, axis=0)
        l_i = l_i * alpha + tl.sum(p, axis=0)
        m_i = m_new

    out = acc / l_i

    tl.store(O + b * stride_ob + h * stride_oh + offs_d, out)


def contiguous_decode_attention(q, k, v, context_lens):
    """
    q: [B, H, D]
    k: [B, H, MAX_SEQ_LEN, D]
    v: [B, H, MAX_SEQ_LEN, D]
    """
    B, H, D = q.shape
    O = torch.empty_like(q)

    grid = (B, H)

    contiguous_decode_attn_kernel[grid](
        q, k, v, O,
        context_lens,
        q.stride(0), q.stride(1),
        k.stride(0), k.stride(1), k.stride(2),
        v.stride(0), v.stride(1), v.stride(2),
        O.stride(0), O.stride(1),
        k.shape[2],
        D,
        BLOCK_N=16,
    )

    return O


def paged_decode_attention(q, k_cache, v_cache, block_table, context_lens, block_size):
    """
    q:           [B, H, D]
    k_cache:    [NUM_BLOCKS, H, BLOCK_SIZE, D]
    v_cache:    [NUM_BLOCKS, H, BLOCK_SIZE, D]
    block_table:[B, MAX_BLOCKS_PER_SEQ]
    """
    B, H, D = q.shape
    O = torch.empty_like(q)

    grid = (B, H)

    paged_decode_attn_kernel[grid](
        q, k_cache, v_cache, block_table, O,
        context_lens,
        q.stride(0), q.stride(1),
        k_cache.stride(0), k_cache.stride(1), k_cache.stride(2),
        v_cache.stride(0), v_cache.stride(1), v_cache.stride(2),
        O.stride(0), O.stride(1),
        block_table.stride(0), block_table.stride(1),
        block_table.shape[1],
        block_size,
        D,
    )

    return O

In [ ]:
# scheduler.py
import math
import random
import torch


class ContiguousKVAllocator:
    """
    Naive KV allocator:
    Every request reserves max_seq_len KV slots.
    """

    def __init__(self, max_num_reqs, max_seq_len):
        self.max_num_reqs = max_num_reqs
        self.max_seq_len = max_seq_len
        self.active_reqs = {}

    def allocate(self, req_id, actual_len):
        # TODO:
        # reject if no empty slot
        if len(self.active_reqs) >= self.max_num_reqs:
            return False

        self.active_reqs[req_id] = {
            "actual_len": actual_len,
            "allocated_tokens": self.max_seq_len,
        }
        return True

    def free(self, req_id):
        # TODO:
        self.active_reqs.pop(req_id, None)

    def allocated_tokens(self):
        return sum(x["allocated_tokens"] for x in self.active_reqs.values())

    def used_tokens(self):
        return sum(x["actual_len"] for x in self.active_reqs.values())

    def waste_ratio(self):
        allocated = self.allocated_tokens()
        used = self.used_tokens()
        if allocated == 0:
            return 0.0
        return 1.0 - used / allocated


class PagedKVAllocator:
    """
    Paged KV allocator:
    Each request only reserves ceil(actual_len / block_size) blocks.
    """

    def __init__(self, num_physical_blocks, block_size, max_blocks_per_seq):
        self.num_physical_blocks = num_physical_blocks
        self.block_size = block_size
        self.max_blocks_per_seq = max_blocks_per_seq

        self.free_blocks = list(range(num_physical_blocks))
        self.req_to_blocks = {}
        self.req_to_actual_len = {}

    def allocate(self, req_id, actual_len):
        # TODO 1:
        # calculate required blocks
        required_blocks = math.ceil(actual_len / self.block_size)

        if required_blocks > self.max_blocks_per_seq:
            return False

        # TODO 2:
        # reject if not enough physical blocks
        if len(self.free_blocks) < required_blocks:
            return False

        # TODO 3:
        # pop physical blocks
        blocks = []
        for _ in range(required_blocks):
            blocks.append(self.free_blocks.pop())

        self.req_to_blocks[req_id] = blocks
        self.req_to_actual_len[req_id] = actual_len
        return True

    def free(self, req_id):
        # TODO:
        blocks = self.req_to_blocks.pop(req_id, [])
        self.req_to_actual_len.pop(req_id, None)
        self.free_blocks.extend(blocks)

    def build_block_table(self, active_req_ids, device="cuda"):
        """
        Return block_table: [B, max_blocks_per_seq]
        """
        table = torch.full(
            (len(active_req_ids), self.max_blocks_per_seq),
            -1,
            dtype=torch.int32,
            device=device,
        )

        # TODO:
        # table[i, j] = physical block id
        for i, req_id in enumerate(active_req_ids):
            blocks = self.req_to_blocks[req_id]
            for j, blk in enumerate(blocks):
                table[i, j] = blk

        return table

    def allocated_tokens(self):
        used_blocks = self.num_physical_blocks - len(self.free_blocks)
        return used_blocks * self.block_size

    def used_tokens(self):
        return sum(self.req_to_actual_len.values())

    def waste_ratio(self):
        allocated = self.allocated_tokens()
        used = self.used_tokens()
        if allocated == 0:
            return 0.0
        return 1.0 - used / allocated

In [ ]:
# scheduler.py
class Request:
    def __init__(self, req_id, arrival_time, prompt_len, output_len):
        self.req_id = req_id
        self.arrival_time = arrival_time
        self.prompt_len = prompt_len
        self.output_len = output_len
        self.generated = 0
        self.start_time = None
        self.finish_time = None

    def is_finished(self):
        return self.generated >= self.output_len


def generate_requests(
    num_requests=512,
    arrival_gap_range=(0, 2),
    prompt_len_range=(64, 1024),
    output_len_range=(16, 256),
    seed=0,
):
    random.seed(seed)

    requests = []
    t = 0

    for i in range(num_requests):
        t += random.randint(*arrival_gap_range)
        prompt_len = random.randint(*prompt_len_range)
        output_len = random.randint(*output_len_range)

        requests.append(
            Request(
                req_id=i,
                arrival_time=t,
                prompt_len=prompt_len,
                output_len=output_len,
            )
        )

    return requests


def run_static_batching(requests, batch_size):
    """
    Static batching:
    1. collect batch_size requests
    2. decode until all finish
    3. then collect next batch
    """

    time = 0
    completed = []
    waiting = list(requests)

    active_batch_sizes = []

    while waiting:
        # TODO 1:
        # take next batch
        batch = waiting[:batch_size]
        waiting = waiting[batch_size:]

        # TODO 2:
        # batch starts when all requests have arrived
        time = max(time, max(r.arrival_time for r in batch))

        for r in batch:
            r.start_time = time

        # TODO 3:
        # decode until all requests finish
        while not all(r.is_finished() for r in batch):
            active = [r for r in batch if not r.is_finished()]
            active_batch_sizes.append(len(active))

            # one decode step
            for r in active:
                r.generated += 1

            time += 1

        for r in batch:
            r.finish_time = time
            completed.append(r)

    return completed, active_batch_sizes


def run_continuous_batching(requests, batch_size):
    """
    Continuous batching:
    At each decoding step:
      - remove finished requests
      - add newly arrived requests if capacity available
      - decode one token for each active request
    """

    time = 0
    completed = []
    waiting = list(requests)
    active = []

    active_batch_sizes = []

    while waiting or active:
        # TODO 1:
        # remove finished
        still_active = []
        for r in active:
            if r.is_finished():
                r.finish_time = time
                completed.append(r)
            else:
                still_active.append(r)
        active = still_active

        # TODO 2:
        # add arrived requests while there is capacity
        while len(active) < batch_size and waiting and waiting[0].arrival_time <= time:
            r = waiting.pop(0)
            r.start_time = time
            active.append(r)

        # If no active request, jump to next arrival
        if not active and waiting:
            time = max(time, waiting[0].arrival_time)
            continue

        active_batch_sizes.append(len(active))

        # TODO 3:
        # one decode step
        for r in active:
            r.generated += 1

        time += 1

    return completed, active_batch_sizes


def summarize_results(completed, active_batch_sizes):
    latencies = [r.finish_time - r.arrival_time for r in completed]
    total_tokens = sum(r.output_len for r in completed)
    total_time = max(r.finish_time for r in completed) - min(r.arrival_time for r in completed)

    avg_latency = sum(latencies) / len(latencies)
    p95_latency = sorted(latencies)[int(0.95 * len(latencies))]
    throughput = total_tokens / total_time
    avg_active_batch = sum(active_batch_sizes) / len(active_batch_sizes)

    return {
        "throughput_tok_per_step": throughput,
        "avg_latency": avg_latency,
        "p95_latency": p95_latency,
        "avg_active_batch": avg_active_batch,
    }

In [ ]:
# benchmark.py
import copy
import pandas as pd

from scheduler import (
    ContiguousKVAllocator,
    PagedKVAllocator,
    generate_requests,
    run_static_batching,
    run_continuous_batching,
    summarize_results,
)


def benchmark_kv_allocator():
    max_seq_len = 2048
    block_size = 16
    max_blocks_per_seq = max_seq_len // block_size

    # Same total KV token budget
    total_kv_token_budget = 2048 * 64

    contiguous = ContiguousKVAllocator(
        max_num_reqs=64,
        max_seq_len=max_seq_len,
    )

    paged = PagedKVAllocator(
        num_physical_blocks=total_kv_token_budget // block_size,
        block_size=block_size,
        max_blocks_per_seq=max_blocks_per_seq,
    )

    requests = generate_requests(
        num_requests=512,
        prompt_len_range=(64, 2048),
        output_len_range=(16, 256),
        seed=42,
    )

    rows = []

    for allocator_name, allocator in [
        ("contiguous", contiguous),
        ("paged", paged),
    ]:
        accepted = 0

        for r in requests:
            actual_len = r.prompt_len + r.output_len
            ok = allocator.allocate(r.req_id, actual_len)

            if ok:
                accepted += 1

            rows.append({
                "allocator": allocator_name,
                "accepted_reqs": accepted,
                "allocated_tokens": allocator.allocated_tokens(),
                "used_tokens": allocator.used_tokens(),
                "waste_ratio": allocator.waste_ratio(),
            })

        # TODO:
        # Optional: free requests according to finish time.
        # For the first version, only test admission capacity.

    df = pd.DataFrame(rows)
    df.to_csv("results/paged_vs_contiguous.csv", index=False)

    print(df.groupby("allocator").tail(1))


def benchmark_batching():
    batch_size = 32

    requests = generate_requests(
        num_requests=512,
        arrival_gap_range=(0, 2),
        prompt_len_range=(64, 1024),
        output_len_range=(16, 256),
        seed=123,
    )

    static_completed, static_active = run_static_batching(
        copy.deepcopy(requests),
        batch_size=batch_size,
    )

    continuous_completed, continuous_active = run_continuous_batching(
        copy.deepcopy(requests),
        batch_size=batch_size,
    )

    static_summary = summarize_results(static_completed, static_active)
    continuous_summary = summarize_results(continuous_completed, continuous_active)

    rows = [
        {"scheduler": "static", **static_summary},
        {"scheduler": "continuous", **continuous_summary},
    ]

    df = pd.DataFrame(rows)
    df.to_csv("results/static_vs_continuous.csv", index=False)

    print(df)


if __name__ == "__main__":
    benchmark_kv_allocator()
    benchmark_batching()

In [ ]:
# plot_results.py
import pandas as pd
import matplotlib.pyplot as plt


def plot():
    kv = pd.read_csv("results/paged_vs_contiguous.csv")
    sched = pd.read_csv("results/static_vs_continuous.csv")

    kv_last = kv.groupby("allocator").tail(1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: KV waste ratio
    axes[0].bar(
        kv_last["allocator"],
        kv_last["waste_ratio"],
    )
    axes[0].set_title("PagedAttention reduces KV cache waste")
    axes[0].set_ylabel("KV waste ratio")
    axes[0].set_xlabel("KV cache allocator")

    # Right: batching throughput
    axes[1].bar(
        sched["scheduler"],
        sched["throughput_tok_per_step"],
    )
    axes[1].set_title("Continuous batching improves throughput")
    axes[1].set_ylabel("Output tokens / step")
    axes[1].set_xlabel("Batching policy")

    plt.tight_layout()
    plt.savefig("results/vllm_micro_results.png", dpi=200)
    plt.show()


if __name__ == "__main__":
    plot()